In [1]:
from pathlib import Path

from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.gmm import GMMPolicy, GMMPolicyConfig
from tapas_gmm.policy.models.tpgmm import AutoTPGMMConfig
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaPlanning

2026-07-06 21:57:10.177 | INFO     |  Running on cpu


/home/nils/Documents/Study Project/Code/riepybdlib/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
model_path = Path(
    "../outputs/bimanual_tpgmm_rotation.pkl"
)

In [3]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode = BimanualEndEffectorPoseViaPlanning,
        robot_setup = "dual_panda",
        task = "BimanualDualPushButtons",
        cameras = tuple(),
        camera_pose = {},
        image_size = (128, 128),
        static = False,
        headless = False,
        scale_action = False,
        delay_gripper = False,
        gripper_plot = False,   
    )
)

policy = GMMPolicy(
    GMMPolicyConfig(
        suffix=None,
        model = AutoTPGMMConfig(),
        batch_predict_in_t_models = False,
        topp_in_t_models = False,
        binary_gripper_action = True,
        force_overwrite_checkpoint_config = True,
        pos_lag_thresh = None,
    )
)

2026-07-06 21:57:17.211 | INFO     |  Initializing Policy:
2026-07-06 21:57:17.211 | INFO     |  No encoder config provided. Using None.
None


In [4]:
obs = tapas_env.reset()

policy.from_disk(str(model_path))
policy.eval()
policy.reset_episode(tapas_env)

2026-07-06 21:57:18.614 | INFO     |  Loading model:
2026-07-06 21:57:18.766 | ERROR    |  Config mismatch
root.tpgmm.add_action_component False != True
root.tpgmm.add_time_component True != False
root.tpgmm.reg_em_finish_diag 0.0002 != 0.001
root.tpgmm.reg_diag_gripper 0.02 != 0.1
root.tpgmm.reg_diag  0.0002 != 0.001
root.tpgmm.reg_em_finish_diag_gripper 0.02 != 0.1
root.tpgmm.reg_init_diag 0.0005 != 5e-05
root.tpgmm.add_gripper_action True != False
root.demos_segmentation.velocity_based True != False
root.demos_segmentation.distance_based False != True
root.demos_segmentation.components_prop_to_len True != False

2026-07-06 21:57:18.766 | WARNING  |  Overwriting config. This can lead to unexpected errors.
2026-07-06 21:57:18.766 | INFO     |  Detected time-based model: True. Using time-driven policy. Set time_based in config to overwrite.
2026-07-06 21:57:18.766 | INFO     |  Creating local marginals
2026-07-06 21:57:18.767 | INFO     |  Changing number of components to 3


In [5]:
import numpy as np

def split_bimanual_action(action):
    left = action[:7]
    right = action[7:14]
    grips = action[14:]
    return left, right, grips

def split_bimanual_pose(ee_pose):
    left = ee_pose[:7]
    right = ee_pose[7:]
    return left, right

def print_step_debug(step, obs, action, policy, reward=None, done=None):
    left_action, right_action, grips = split_bimanual_action(action)
    left_pose, right_pose = split_bimanual_pose(obs.ee_pose.numpy())

    active_segment = getattr(policy.model, "_online_active_segment", None)
    if active_segment is not None and policy.model.segment_frame_views is not None:
        frame_names = policy.model.segment_frame_views[active_segment].frame_names
    else:
        frame_names = None

    print("\n==============================")
    print("step:", step)
    print("t_curr:", policy._t_curr)
    print("active_segment:", active_segment)
    print("segment_frames:", frame_names)

    print("left current pose:", left_pose)
    print("right current pose:", right_pose)

    print("left action:", left_action)
    print("right action:", right_action)
    print("grips left/right:", grips)

    print("left pos delta norm:", np.linalg.norm(left_action[:3]))
    print("right pos delta norm:", np.linalg.norm(right_action[:3]))

    print("left quat norm:", np.linalg.norm(left_action[3:7]))
    print("right quat norm:", np.linalg.norm(right_action[3:7]))

    if reward is not None:
        print("reward:", reward)
    if done is not None:
        print("done:", done)

In [6]:
done = False
total_reward = 0

for step in range(300):
    action, info = policy.predict(obs)

    if policy._t_curr is not None and float(policy._t_curr[0]) > 1.0:
        print("stopping: model time finished", policy._t_curr)
        break

    print_step_debug(step, obs, action, policy)
    old_left = obs.ee_pose.numpy()[:3].copy()
    old_right = obs.ee_pose.numpy()[7:10].copy()
    obs, reward, done, env_info = tapas_env.step(action)
    new_left = obs.ee_pose.numpy()[:3]
    new_right = obs.ee_pose.numpy()[7:10]

    print("left moved:", np.linalg.norm(new_left - old_left))
    print("right moved:", np.linalg.norm(new_right - old_right))
    total_reward += reward

    print("after step reward:", reward)
    print("after step done:", done)
    print("env_info:", env_info)

    if done or info.get("done", False):
        break

print("steps:", step + 1)
print("total_reward:", total_reward)

2026-07-06 21:57:18.802 | WARNING  |  Implementation lacking modulo rots, enforce z-up/down, etc.
2026-07-06 21:57:18.996 | WARNING  |  Product did not converge in 50 iterations.

step: 0
t_curr: [0.01515152]
active_segment: 0
segment_frames: ('left world', 'left ee_init', 'right world', 'right ee_init')
left current pose: [ 0.3020686   0.06809669  1.4714357   0.12045165 -0.04330131  0.99176055
  0.00525144]
right current pose: [ 2.45577767e-01 -2.41837487e-01  1.47210455e+00  5.38759332e-06
 -9.92670119e-01 -6.28731627e-08  1.20855756e-01]
left action: [ 0.47674782  0.01289022  1.41450626  0.99998574 -0.00154623 -0.00489202
 -0.00159681]
right action: [ 0.18011497 -0.24182362  1.47981105  0.03055013 -0.06659185 -0.83308129
 -0.54827725]
grips left/right: [1. 1.]
left pos delta norm: 1.4927433192267932
right pos delta norm: 1.5102187940371459
left quat norm: 1.0000001734648387
right quat norm: 1.0000000785882686
2026-07-06 21:57:19.242 | INFO     |  Action [ 0.18011497 -0.24182362  1.4

In [7]:
tapas_env.close()

[CoppeliaSim:loadinfo]   done.
